In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableMap

/tmp/ipykernel_3101/2799209537.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
# Load the textfile
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()

# Split document into chunks
splitter = RecursiveCharacterTextSplitter(
    separators=["."],
    chunk_size = 300,chunk_overlap = 50
)
docs = splitter.split_documents(raw_docs)
docs

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain and CrewAI represent two complementary paradigms for building applications with large \nlanguage models, and understanding how they fit together is essential for anyone designing \nmodern AI agent systems; LangChain, at its core, is a framework for composing chains of \noperations around LLMs, providing abstractions for models, prompts, memory, retrievers, tools, \nand output parsers that let developers wire together deterministic pipelines where each step \ntransforms data before passing it to the next, and its ecosystem includes LangGraph for stateful, \ngraph-based orchestration, LangSmith for tracing and observability, and LangServe for deploying chains \nas APIs, making it ideal for scenarios where you need fine-grained control over the flow of data, \nretrieval-augmented generation, document processing, conversational memory, and integration with dozens \nof vector stores, APIs, and data source

In [3]:
embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorstores = FAISS.from_documents(docs,embedding_model)
retriever = vectorstores.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":3}
)
retriever

/tmp/ipykernel_3101/3719169131.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7fc01c4407d0>, search_type='mmr', search_kwargs={'k': 3})

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
llm = ChatGroq(
     model="openai/gpt-oss-20b",
     temperature=0.7,
     api_key=os.getenv("GROQ_API_KEY")
 )

In [11]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
   You are a helpful assistant now Expand the query. Preserve intent.

QUERY: "{query}"

Generate [N] variants: include acronym expansion + synonyms.
Max [MAX_WORDS] words each. JSON only.

{{"original": "<query>", "expanded": ["<q1>", "<q2>"]}}
""")
query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\n   You are a helpful assistant now Expand the query. Preserve intent.\n\nQUERY: "{query}"\n\nGenerate [N] variants: include acronym expansion + synonyms.\nMax [MAX_WORDS] words each. JSON only.\n\n{{"original": "<query>", "expanded": ["<q1>", "<q2>"]}}\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7fc013c52c30>, async_clien

In [12]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'{"original":"Langchain memory","expanded":["LangChain memory component","LangChain memory system","LangChain memory module","LangChain memory feature","Language Chain memory"]}'

In [13]:
prompt = PromptTemplate.from_template("""
    Answer the question based on the context below.
    
    context:{context}
    Question:{input}"""
)
document_chain = create_stuff_documents_chain(llm,prompt)


In [14]:
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x:retriever.invoke(query_expansion_chain.invoke({"query":x["input"]}))
    }) | document_chain
)

In [15]:
query = {"input":"What types of memory does langchain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("Answer: \n",response)

{"original":"What types of memory does langchain support?","expanded":["What kinds of memory does LangChain (LC) support?","Which memory types does LangChain (Language Chain) provide?","What memory options does LangChain support?","What forms of memory does LangChain (LC) offer?","Which memory mechanisms are available in LangChain?"]}
Answer: 
 LangChain ships with a family of **memory back‑ends** that let you persist and retrieve conversational context across turns.  The core types are:

| Memory type | What it stores | Typical use case |
|-------------|----------------|------------------|
| **ConversationBufferMemory** | The entire chat history (all turns) | Simple, unlimited history – good for debugging or when you don’t care about token limits. |
| **ConversationBufferWindowMemory** | The last *N* turns (a sliding window) | Keeps recent context while discarding older turns; useful when you want to limit memory size. |
| **ConversationSummaryMemory** | A summarized version of the wh